# **Análisis de ofertas de empleo de Konzerta**

**Curso:** Introducción a la Programación para Ciencia de Datos

**Integrantes:** Francia Vergara, Luis Santimateo y Yenia Rojas

**Fuente:** Konzerta

**Archivo utilizado:** Konzerta 08-oct-2026.xlsx

**Fecha de obtención del archivo:** 16-oct-2026



###**Contexto del proyecto**

En este proyecto buscamos conocer las características de las oportunidades de empleo incluidas en el archivo de Konzerta. Nos interesa identificar qué áreas profesionales concentran más anuncios y vacantes, dónde se ubican las oportunidades y qué modalidades de trabajo ofrecen.

También exploraremos la información salarial, su relación con la experiencia solicitada y la cantidad de anuncios publicados por semana. Para interpretar estos resultados, tendremos en cuenta los datos faltantes y los valores que requieren revisión.

###**Objetivo del análisis**

Analizar las ofertas de empleo de Konzerta mediante la revisión, limpieza y exploración de los datos, para responder las preguntas de nuestra propuesta y comunicar los hallazgos con tablas y gráficos.

El alcance del análisis corresponde a los anuncios incluidos en el archivo utilizado.

###**Preguntas de análisis**

¿Qué áreas y subáreas profesionales presentan mayor demanda según las
ofertas y vacantes publicadas?

¿Cómo varían los salarios ofrecidos según el nivel de experiencia requerido en las ofertas que cuentan con información salarial?

¿Qué modalidades de trabajo son más frecuentes y cómo cambian según el área?

¿Cómo se distribuyen las oportunidades laborales según área profesional, nivel laboral y ubicación geográfica?

¿Cómo evoluciona el volumen de ofertas publicadas por semana entre julio y octubre de 2026?

In [112]:
# CONFIGURACIÓN DEL ENTORNO
# Compatible con Google Colab, Windows y macOS

from pathlib import Path
import sys

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROYECTO = Path("/content/drive/MyDrive/Proyecto_Konzerta")
else:
    actual = Path.cwd().resolve()

    PROYECTO = next(
        (
            ruta for ruta in [actual, *actual.parents]
            if ruta.name == "Proyecto_Konzerta"
        ),
        None
    )

    if PROYECTO is None:
        candidato = actual / "Proyecto_Konzerta"
        if candidato.is_dir():
            PROYECTO = candidato
        else:
            raise FileNotFoundError(
                "Abre JupyterLab desde Proyecto_Konzerta."
            )

BRONCE = PROYECTO / "datos" / "bronce"
PLATA = PROYECTO / "datos" / "plata"
ORO = PROYECTO / "datos" / "oro"
NOTEBOOKS = PROYECTO / "notebooks"
DASHBOARD = PROYECTO / "dashboard"

for carpeta in [BRONCE, PLATA, ORO, NOTEBOOKS, DASHBOARD]:
    carpeta.mkdir(parents=True, exist_ok=True)

(PROYECTO / "README.md").touch(exist_ok=True)
(PROYECTO / "requirements.txt").touch(exist_ok=True)

print("Entorno:", "Google Colab" if EN_COLAB else "Local")
print("Proyecto:", PROYECTO)
print("Configuración completada correctamente.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entorno: Google Colab
Proyecto: /content/drive/MyDrive/Proyecto_Konzerta
Configuración completada correctamente.


# **Lectura y revisión inicial de los datos**

Leemos el archivo de Konzerta y conservamos una tabla original como referencia. Después creamos una copia para trabajar y comparar los cambios realizados durante la limpieza.

Cada fila representa un anuncio de empleo. Revisamos el tamaño del archivo, sus primeros registros y los tipos de datos para conocer su estructura antes de comenzar la limpieza.

In [84]:
# CAPA BRONCE: LECTURA DEL ARCHIVO ORIGINAL

import pandas as pd

archivo_excel = BRONCE / "Konzerta 08-oct-2026.xlsx"

if not archivo_excel.is_file():
    raise FileNotFoundError(
        f"No se encontró el archivo: {archivo_excel}"
    )

crudos = pd.read_excel(archivo_excel)
konzerta = crudos.copy()

print(f"Total de anuncios: {crudos.shape[0]:,}")
print(f"Total de columnas: {crudos.shape[1]}")

display(
    konzerta[
        ["id", "titulo", "area", "provincia", "modalidad"]
    ].head(3)
)

konzerta.info()

Total de anuncios: 3,011
Total de columnas: 31


,id,titulo,area,provincia,modalidad
0,1118469088,Ortodoncista,"Salud, Medicina y Farmacia",Panamá,Presencial
1,1118473088,Gerente de Auditoría Externa,"Administración, Contabilidad y Finanzas",Panamá,Presencial
2,1118473021,Ejecutivo de Atencion al Cliente - Canales Dig...,"Atención al Cliente, Call Center y Telemarketing",Panamá,Presencial


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3011 entries, 0 to 3010
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id                    3011 non-null   int64         
 1   titulo                3011 non-null   object        
 2   empresa               1846 non-null   object        
 3   empresa_confidencial  3011 non-null   bool          
 4   industria             1788 non-null   object        
 5   empresa_tamano        1820 non-null   object        
 6   fecha_publicacion     3011 non-null   datetime64[ns]
 7   fecha_finalizacion    3011 non-null   datetime64[ns]
 8   provincia             3011 non-null   object        
 9   localidad             2969 non-null   object        
 10  area                  3011 non-null   object        
 11  subarea               3011 non-null   object        
 12  nivel_laboral         2758 non-null   object        
 13  tipo_trabajo      

## **Revisión de valores faltantes**

Revisamos cuántos datos faltan en cada columna y qué porcentaje representan sobre el total de anuncios. Esto nos ayuda a reconocer las limitaciones de la información y a decidir cómo tratarla durante la limpieza.

Las decisiones sobre estos faltantes se documentan en la bitácora para explicar qué hicimos y por qué.

In [85]:
# Medimos los faltantes sobre la tabla original, antes de la limpieza
total_anuncios = len(crudos)
faltantes_por_columnas = crudos.isna().sum()

porcentaje_por_columna = (faltantes_por_columnas / total_anuncios * 100).round(2)

# Reunimos cantidades y porcentajes en una sola tabla
resumen_faltante = pd.DataFrame({"cantidad_faltante": faltantes_por_columnas, "porcentaje_faltante": porcentaje_por_columna}).sort_values(by="cantidad_faltante", ascending=False)
display(resumen_faltante)

,cantidad_faltante,porcentaje_faltante
idiomas,2606,86.55
salario_min,2546,84.56
salario_tipo,2526,83.89
salario_max,2526,83.89
salario_frecuencia,2526,83.89
edad_min,2344,77.85
edad_max,2344,77.85
experiencia_meses,2159,71.70
educacion,1789,59.42
educacion_estado,1789,59.42


# **Revisión de duplicados**

Comprobamos si hay filas completamente iguales y si algún identificador de anuncio aparece más de una vez. Estas repeticiones podrían alterar los conteos de ofertas y vacantes.

Revisamos ambas situaciones porque un anuncio puede conservar el mismo identificador y tener diferencias en otros campos. Las decisiones sobre su tratamiento se documentan en la bitácora de limpieza.

In [86]:
# Buscamos repeticiones en los datos originales
duplicados_completos = crudos.duplicated().sum()
duplicados_id = crudos["id"].duplicated().sum()

print("Filas completamente duplicadas:", duplicados_completos)
print("Repeticiones de id:", duplicados_id)

Filas completamente duplicadas: 0
Repeticiones de id: 0


**No se encontraron filas completamente duplicadas ni identificadores repetidos en los datos originales. Por este motivo, no es necesario eliminar anuncios por duplicación.**

# **Revisión de valores numéricos**

Revisamos las vacantes, los salarios, las edades y la experiencia para conocer sus valores habituales e identificar cifras que necesitan validación.

Primero consultamos las estadísticas de cada variable, como el promedio, la mediana y los valores mínimos y máximos. Después observamos los cinco anuncios con mayores salarios máximos registrados.

Esta revisión sirve para orientar la limpieza. Un valor extremo requiere evaluación antes de decidir su tratamiento. Además, la moneda de los importes salariales sigue pendiente de confirmar, lo que limita su comparación entre anuncios.

In [87]:
# Seleccionamos las variables numéricas relevantes para el análisis
columnas_numericas = ["vacantes", "salario_min", "salario_max", "edad_min", "edad_max", "experiencia_meses"]
resumen_numerico = crudos[columnas_numericas].describe().T

# Redondeamos la presentación para facilitar la lectura
display(resumen_numerico.round(2))

# Revisamos los anuncios con mayores importes salariales registrados
columnas_salario = ["id", "titulo", "salario_min", "salario_max", "salario_frecuencia", "salario_tipo"]

display(crudos[columnas_salario].sort_values(by="salario_max", ascending=False).head(5))

,count,mean,std,min,25%,50%,75%,max
vacantes,3011.0,2.71,6.79,1.0,1.0,1.0,1.0,50.0
salario_min,465.0,2860.38,41710.19,1.0,630.0,750.0,1000.0,900000.0
salario_max,485.0,4278.46,61394.19,0.0,700.0,980.0,1450.0,1350000.0
edad_min,667.0,24.79,4.61,18.0,20.0,25.0,27.0,45.0
edad_max,667.0,42.14,7.34,23.0,36.0,43.0,45.0,99.0
experiencia_meses,852.0,28.32,18.46,12.0,12.0,24.0,36.0,120.0


,id,titulo,salario_min,salario_max,salario_frecuencia,salario_tipo
2081,1118419642,Desarrollador Junior Full Stack (Móvil y Web c...,900000.0,1350000.0,por Mes,BRUTO
1259,1118440234,Agente Comercial (Panama Metro),NaN,72176.0,por Mes,BRUTO
1751,1118428715,ASESOR DE VENTAS,NaN,62816.0,por Mes,BRUTO
739,1118452388,Director(a) de Logística Integral – México y P...,19900.0,20000.0,por Mes,BRUTO
1923,1118423515,Asistente Contable y Administrativo,12000.0,12001.0,por Mes,BRUTO


# **Limpieza de datos**

Aplicamos los ajustes identificados durante la revisión inicial. Cada ejecución comienza con una nueva copia de los datos originales para que podamos repetir el proceso y comparar los cambios.

Primero quitamos los espacios sobrantes en los nombres de empresa y convertimos la fecha de extracción a un formato de fecha y hora. Guardamos el perfil inicial de los datos y registramos cada decisión en la bitácora de limpieza.

In [88]:
# CAPA PLATA: NORMALIZACIÓN DE DATOS

import re
import unicodedata

konzerta = crudos.copy(deep=True)

perfil_antes = pd.DataFrame({
    "tipo": konzerta.dtypes.astype(str),
    "faltantes": konzerta.isna().sum()
})

registros_bitacora = []

def registrar(columna, problema, decision, filas_afectadas):
    registros_bitacora.append({
        "columna": columna,
        "problema": problema,
        "decision": decision,
        "filas_afectadas": int(filas_afectadas)
    })

# Normalizar nombres de columnas
def normalizar_columna(nombre):
    nombre = unicodedata.normalize("NFKD", str(nombre))
    nombre = "".join(
        c for c in nombre
        if not unicodedata.combining(c)
    )
    nombre = re.sub(r"[^a-zA-Z0-9]+", "_", nombre)
    return nombre.strip("_").lower()

nombres_anteriores = konzerta.columns.tolist()
konzerta.columns = [
    normalizar_columna(c) for c in konzerta.columns
]

registrar(
    "nombres_columnas",
    "Posibles nombres sin formato uniforme",
    "Normalizar a snake_case",
    sum(a != b for a, b in zip(
        nombres_anteriores, konzerta.columns
    ))
)

# Limpiar columnas de texto
columnas_texto = konzerta.select_dtypes(
    include=["object", "string"]
).columns

for columna in columnas_texto:
    if columna in ["descripcion", "url"]:
        continue

    original = konzerta[columna].astype("string")

    limpio = (
        original.str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )

    cambios = original.fillna("").ne(
        limpio.fillna("")
    ).sum()

    konzerta[columna] = limpio

    if cambios > 0:
        registrar(
            columna,
            "Espacios sobrantes o textos vacíos",
            "Normalizar espacios y convertir vacíos a nulos",
            cambios
        )

# Convertir fechas
for columna in [
    "fecha_publicacion",
    "fecha_finalizacion",
    "extraido"
]:
    if columna in konzerta.columns:
        antes = konzerta[columna].notna()

        konzerta[columna] = pd.to_datetime(
            konzerta[columna],
            errors="coerce"
        )

        registrar(
            columna,
            "Fechas no interpretables",
            "Convertir a datetime",
            (antes & konzerta[columna].isna()).sum()
        )

# Convertir números
for columna in [
    "vacantes", "salario_min", "salario_max",
    "edad_min", "edad_max", "experiencia_meses"
]:
    if columna in konzerta.columns:
        antes = konzerta[columna].notna()

        konzerta[columna] = pd.to_numeric(
            konzerta[columna],
            errors="coerce"
        )

        registrar(
            columna,
            "Valores numéricos no interpretables",
            "Convertir a tipo numérico",
            (antes & konzerta[columna].isna()).sum()
        )

print("Normalización completada.")
print("Filas:", len(konzerta))
print("Columnas:", len(konzerta.columns))

Normalización completada.
Filas: 3011
Columnas: 31


# **Tratamiento de faltantes y revisión de salarios**

En las categorías que utilizaremos para agrupar los anuncios, representamos la información ausente con la etiqueta “Sin dato”. Los faltantes numéricos se conservan como valores ausentes.

Para los salarios, aplicamos reglas que permiten identificar importes que requieren revisión. También utilizamos el rango intercuartílico (IQR), que mide la dispersión de la mitad central de los salarios máximos registrados.

**Clasificamos los anuncios en tres grupos:**

**Sin dato:** no tienen salario mínimo ni máximo registrado.

**Registro sin alerta:** tienen algún importe registrado y no activan las reglas aplicadas.

**Revisar:** presentan algún importe o rango que necesita validación.

Los importes se conservan. Estas categorías reflejan las comprobaciones realizadas; la moneda sigue pendiente de confirmar.

In [89]:
# CAPA PLATA: FALTANTES, DUPLICADOS Y SALARIOS

categorias = [
    "area", "subarea", "provincia",
    "localidad", "nivel_laboral", "modalidad"
]

for columna in categorias:
    faltantes = int(konzerta[columna].isna().sum())

    konzerta[columna] = (
        konzerta[columna]
        .astype("string")
        .fillna("Sin dato")
    )

    registrar(
        columna,
        "Valores faltantes en categorías",
        "Reemplazar nulos por Sin dato",
        faltantes
    )

# Eliminar duplicados completos
duplicados_eliminados = int(konzerta.duplicated().sum())

konzerta = konzerta.drop_duplicates().copy()

registrar(
    "todas_las_columnas",
    "Filas completamente duplicadas",
    "Eliminar registros idénticos",
    duplicados_eliminados
)

# Revisar valores salariales mediante IQR
q1 = konzerta["salario_max"].quantile(0.25)
q3 = konzerta["salario_max"].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

alerta_salario = (
    (konzerta["salario_min"] <= 1)
    | (konzerta["salario_max"] <= 0)
    | (konzerta["salario_min"] > konzerta["salario_max"])
    | (konzerta["salario_max"] < limite_inferior)
    | (konzerta["salario_max"] > limite_superior)
)

con_salario = (
    konzerta["salario_min"].notna()
    | konzerta["salario_max"].notna()
)

konzerta["estado_salario"] = "Sin dato"

konzerta.loc[
    con_salario, "estado_salario"
] = "Registro sin alerta"

konzerta.loc[
    con_salario & alerta_salario, "estado_salario"
] = "Revisar"

cantidad_alertas = int(
    (konzerta["estado_salario"] == "Revisar").sum()
)

registrar(
    "estado_salario",
    "Importes inconsistentes o extremos",
    "Conservar salarios y marcar para revisión",
    cantidad_alertas
)

print("Duplicados eliminados:", duplicados_eliminados)
print("Alertas salariales:", cantidad_alertas)

display(konzerta["estado_salario"].value_counts())

Duplicados eliminados: 0
Alertas salariales: 71


,count
estado_salario,
Sin dato,2526
Registro sin alerta,414
Revisar,71


In [90]:
# CAPA PLATA: BITÁCORA Y PERFIL DE CALIDAD

for columna in konzerta.columns:

    if columna == "estado_salario":
        continue

    cantidad = int(konzerta[columna].isna().sum())

    if cantidad > 0:
        registrar(
            columna,
            "Valores faltantes después de la limpieza",
            "Conservar los nulos sin inventar información",
            cantidad
        )

bitacora = pd.DataFrame(
    registros_bitacora,
    columns=[
        "columna",
        "problema",
        "decision",
        "filas_afectadas"
    ]
)

perfil_despues = pd.DataFrame({
    "tipo": konzerta.dtypes.astype(str),
    "faltantes": konzerta.isna().sum(),
    "sin_dato": konzerta.eq("Sin dato").sum()
})

print("Registros originales:", len(crudos))
print("Registros limpios:", len(konzerta))
print("Duplicados eliminados:", duplicados_eliminados)

print("\nBITÁCORA DE LIMPIEZA")
display(bitacora)

print("\nPERFIL DE CALIDAD")
display(perfil_despues)

Registros originales: 3011
Registros limpios: 3011
Duplicados eliminados: 0

BITÁCORA DE LIMPIEZA


,columna,problema,decision,filas_afectadas
0,nombres_columnas,Posibles nombres sin formato uniforme,Normalizar a snake_case,0
1,empresa,Espacios sobrantes o textos vacíos,Normalizar espacios y convertir vacíos a nulos,122
2,localidad,Espacios sobrantes o textos vacíos,Normalizar espacios y convertir vacíos a nulos,14
3,subarea,Espacios sobrantes o textos vacíos,Normalizar espacios y convertir vacíos a nulos,65
4,fecha_publicacion,Fechas no interpretables,Convertir a datetime,0
5,fecha_finalizacion,Fechas no interpretables,Convertir a datetime,0
6,extraido,Fechas no interpretables,Convertir a datetime,0
7,vacantes,Valores numéricos no interpretables,Convertir a tipo numérico,0
8,salario_min,Valores numéricos no interpretables,Convertir a tipo numérico,0
9,salario_max,Valores numéricos no interpretables,Convertir a tipo numérico,0



PERFIL DE CALIDAD


,tipo,faltantes,sin_dato
id,int64,0,0
titulo,string,0,0
empresa,string,1165,0
empresa_confidencial,bool,0,0
industria,string,1223,0
empresa_tamano,string,1191,0
fecha_publicacion,datetime64[ns],0,0
fecha_finalizacion,datetime64[ns],0,0
provincia,string,0,0
localidad,string,0,42


# **Verificación de la limpieza**

Comparamos la cantidad de filas y columnas, los tipos de datos y los valores faltantes antes y después de la limpieza.

La columna sin_dato cuenta las etiquetas que representan información ausente. Esto permite distinguirlas de los valores nulos que conservamos.

Finalmente, presentamos la bitácora completa para explicar qué cambios realizamos y cuál fue su justificación.

In [91]:
# CAPA PLATA: VERIFICACIÓN FINAL

comparacion_dimensiones = pd.DataFrame({
    "filas": [crudos.shape[0], konzerta.shape[0]],
    "columnas": [crudos.shape[1], konzerta.shape[1]]
}, index=["Antes", "Después"])

display(comparacion_dimensiones)

comparacion_perfiles = perfil_antes.join(
    perfil_despues,
    how="outer",
    lsuffix="_antes",
    rsuffix="_despues"
)

display(comparacion_perfiles)

duplicados_restantes = int(konzerta.duplicated().sum())

columnas_esenciales = [
    "id", "titulo", "area", "subarea",
    "provincia", "modalidad",
    "fecha_publicacion", "vacantes"
]

faltan_columnas = [
    c for c in columnas_esenciales
    if c not in konzerta.columns
]

if faltan_columnas:
    raise ValueError(
        f"Faltan columnas esenciales: {faltan_columnas}"
    )

print("Duplicados restantes:", duplicados_restantes)
print("Columnas esenciales verificadas.")
print("Registros procesados:", len(konzerta))
print("Registros de bitácora:", len(bitacora))

,filas,columnas
Antes,3011,31
Después,3011,32


,tipo_antes,faltantes_antes,tipo_despues,faltantes_despues,sin_dato
apto_discapacidad,bool,0.0,bool,0,0
area,object,0.0,string,0,0
contratacion,object,598.0,string,598,0
descripcion,object,0.0,object,0,0
edad_max,float64,2344.0,float64,2344,0
edad_min,float64,2344.0,float64,2344,0
educacion,object,1789.0,string,1789,0
educacion_estado,object,1789.0,string,1789,0
empresa,object,1165.0,string,1165,0
empresa_confidencial,bool,0.0,bool,0,0


Duplicados restantes: 0
Columnas esenciales verificadas.
Registros procesados: 3011
Registros de bitácora: 35


In [92]:
# CAPA PLATA: EXPORTAR RESULTADOS

PLATA.mkdir(parents=True, exist_ok=True)

konzerta.to_csv(
    PLATA / "konzerta_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

bitacora.to_csv(
    PLATA / "bitacora_limpieza.csv",
    index=False,
    encoding="utf-8-sig"
)

perfil_antes.to_csv(
    PLATA / "perfil_bronce.csv",
    encoding="utf-8-sig"
)

perfil_despues.to_csv(
    PLATA / "perfil_plata.csv",
    encoding="utf-8-sig"
)

print("Archivos de la capa plata generados:")

for archivo in sorted(PLATA.glob("*.csv")):
    print(archivo.name)

Archivos de la capa plata generados:
bitacora_limpieza.csv
konzerta_limpio.csv
perfil_bronce.csv
perfil_plata.csv


El proceso estandariza los espacios en los nombres de empresa, convierte la fecha de extracción y añade una clasificación de la información salarial. En las categorías seleccionadas, la etiqueta “Sin dato” permite incluir los anuncios con información incompleta. Los faltantes numéricos se conservan como valores ausentes.

En estado_salario, los campos del perfil anterior aparecen como NaN porque esa columna se creó durante la limpieza.

Los importes marcados como “Revisar” mantienen sus valores originales. Su clasificación indica que requieren validación antes de interpretar las diferencias salariales

**Fuente:** [Konzerta](https://www.konzerta.com/)

# **Diccionario de datos**

Este diccionario explica las columnas que utilizaremos en el análisis. Los tipos de datos corresponden a la tabla después de la limpieza.

Incluimos las 31 columnas originales y estado_salario, que creamos para distinguir la disponibilidad de importes salariales y los casos que requieren revisión.

In [93]:
# Explicamos el significado de cada columna
descripciones = {"id": "Identificador del anuncio de empleo.",
    "titulo": "Nombre del puesto publicado.",
    "empresa": "Nombre de la empresa anunciante, cuando se informa.",
    "empresa_confidencial": "Indica si la empresa se presenta como confidencial.",
    "industria": "Sector de actividad registrado para la empresa.",
    "empresa_tamano": "Categoría de tamaño de la empresa.",
    "fecha_publicacion": "Fecha de publicación del anuncio.",
    "fecha_finalizacion": "Fecha de finalización indicada para el anuncio.",
    "provincia": "Provincia registrada para la oferta.",
    "localidad": "Localidad registrada para la oferta.",
    "area": "Área profesional a la que pertenece el anuncio.",
    "subarea": "Subárea o especialidad profesional del anuncio.",
    "nivel_laboral": "Nivel o jerarquía del puesto.",
    "tipo_trabajo": "Clasificación del trabajo, como Full-time o Part-time.",
    "modalidad": "Forma de trabajo: presencial, híbrida o remota.",
    "contratacion": "Tipo de contratación, como indeterminado o temporal.",
    "vacantes": "Cantidad de puestos ofrecidos en el anuncio.",
    "salario_min": "Importe salarial mínimo registrado; moneda sin confirmar.",
    "salario_max": "Importe salarial máximo registrado; moneda sin confirmar.",
    "salario_frecuencia": "Periodicidad del salario registrado, por ejemplo mensual.",
    "salario_tipo": "Tipo de importe salarial registrado, por ejemplo bruto.",
    "edad_min": "Edad mínima solicitada, en años.",
    "edad_max": "Edad máxima solicitada, en años.",
    "experiencia_meses": "Experiencia requerida registrada en meses.",
    "educacion": "Nivel educativo solicitado.",
    "educacion_estado": "Situación de los estudios, como graduado o en curso.",
    "idiomas": "Idiomas y niveles de dominio indicados en el anuncio.",
    "apto_discapacidad": "Indica si el anuncio señala que es apto para personas con discapacidad.",
    "descripcion": "Texto del anuncio con información del puesto y sus requisitos.",
    "url": "Enlace al anuncio original.",
    "extraido": "Fecha y hora de extracción registradas en el dataset.",
    "estado_salario": "Clasificación creada en la limpieza según disponibilidad de importes y alertas."}

# Obtenemos los tipos directamente de la tabla limpia
diccionario_datos = pd.DataFrame({"columna": konzerta.columns,"tipo": list(konzerta.dtypes.astype(str))})

diccionario_datos["descripcion"] = (diccionario_datos["columna"].map(descripciones))

display(diccionario_datos.style.set_properties(**{"text-align": "left"}).set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]}]))

,columna,tipo,descripcion
0,id,int64,Identificador del anuncio de empleo.
1,titulo,string,Nombre del puesto publicado.
2,empresa,string,"Nombre de la empresa anunciante, cuando se informa."
3,empresa_confidencial,bool,Indica si la empresa se presenta como confidencial.
4,industria,string,Sector de actividad registrado para la empresa.
5,empresa_tamano,string,Categoría de tamaño de la empresa.
6,fecha_publicacion,datetime64[ns],Fecha de publicación del anuncio.
7,fecha_finalizacion,datetime64[ns],Fecha de finalización indicada para el anuncio.
8,provincia,string,Provincia registrada para la oferta.
9,localidad,string,Localidad registrada para la oferta.


# **Análisis exploratorio de datos**

###**Pregunta 1. Áreas y subáreas con mayor demanda**

¿Qué áreas y subáreas profesionales presentan mayor demanda según las ofertas y vacantes publicadas?

Comenzamos agrupando los anuncios por área profesional. Contaremos las ofertas y sumaremos las vacantes registradas.

Una oferta corresponde a un anuncio; las vacantes representan los puestos que ese anuncio ofrece. Esta diferencia permite observar tanto la cantidad de publicaciones como los puestos anunciados.

In [94]:
# Contamos los anuncios y sumamos sus vacantes por área profesional
tabla_areas = (konzerta.groupby("area", dropna=False).agg(ofertas=("id", "size"),vacantes=("vacantes", "sum")).reset_index().sort_values(by="ofertas", ascending=False))

# Calculamos la participación de cada área en el total de anuncios
tabla_areas["porcentaje_ofertas"] = (tabla_areas["ofertas"] / len(konzerta) * 100).round(2)

display(tabla_areas)

,area,ofertas,vacantes,porcentaje_ofertas
4,"Comercial, Ventas y Negocios",801,2808,26.60
1,"Administración, Contabilidad y Finanzas",446,625,14.81
0,Abastecimiento y Logística,294,626,9.76
17,Oficios y Otros,225,617,7.47
24,"Tecnología, Sistemas y Telecomunicaciones",148,318,4.92
3,"Atención al Cliente, Call Center y Telemarketing",146,1608,4.85
9,Gastronomía y Turismo,143,314,4.75
18,Producción y Manufactura,142,268,4.72
12,Ingenierías,118,166,3.92
11,Ingeniería Civil y Construcción,99,166,3.29


In [95]:
import plotly.express as px

# Seleccionamos las diez áreas con más anuncios
top_areas = tabla_areas.head(10).sort_values(by="ofertas")

fig = px.bar(top_areas, x="ofertas", y="area", orientation="h", text="ofertas", title="Las 10 áreas con más ofertas en el dataset de Konzerta",
    labels={"ofertas": "Número de ofertas (anuncios)" ,"area": "Área profesional"})

fig.update_layout(height=650)
fig.show()

Comercial, Ventas y Negocios encabezan las publicaciones del dataset con 801 anuncios, equivalentes al 26.60 % del total. Aunque Administración tiene más anuncios (446) que Atención al Cliente (146), esta última reúne más vacantes (1,608 frente a 625), lo que refleja una mayor cantidad de puestos por anuncio en promedio.

### **Análisis de subáreas profesionales con mayor demanda**

Dentro de las áreas identificadas, existen subáreas o especialidades específicas. Vamos a agrupar los datos por `area` y `subarea` para analizar cuáles son las disciplinas específicas más solicitadas en la plataforma de Konzerta.

In [96]:
# Agrupamos por área y subárea para contar ofertas y sumar vacantes
tabla_subareas = (
    konzerta.groupby(["area", "subarea"], dropna=False)
    .agg(ofertas=("id", "size"), vacantes=("vacantes", "sum"))
    .reset_index()
    .sort_values(by="ofertas", ascending=False)
)

# Calculamos la participación porcentual de cada subárea sobre el total general
tabla_subareas["porcentaje_ofertas"] = (tabla_subareas["ofertas"] / len(konzerta) * 100).round(2)

# Mostramos las 15 subáreas con mayor número de ofertas
display(tabla_subareas.head(15))

,area,subarea,ofertas,vacantes,porcentaje_ofertas
40,"Comercial, Ventas y Negocios",Ventas,384,1077,12.75
34,"Comercial, Ventas y Negocios",Comercial,343,1625,11.39
12,"Administración, Contabilidad y Finanzas",Contabilidad,144,157,4.78
8,"Administración, Contabilidad y Finanzas",Administración,118,138,3.92
1,Abastecimiento y Logística,Almacén / Depósito / Expedición,96,202,3.19
30,"Atención al Cliente, Call Center y Telemarketing",Atención al Cliente,94,966,3.12
57,Gastronomía y Turismo,Gastronomia,78,203,2.59
5,Abastecimiento y Logística,Logística,77,145,2.56
113,Oficios y Otros,Otros,71,233,2.36
121,Producción y Manufactura,Mantenimiento,63,79,2.09


### **Visualización de las subáreas más demandadas**

Para facilitar la comprensión visual de las especialidades más buscadas, creamos un gráfico de barras horizontales con las 10 subáreas que concentran más ofertas en el dataset.

In [97]:
# Seleccionamos las diez subáreas líderes
top_subareas = tabla_subareas.head(10).sort_values(by="ofertas")

fig_sub = px.bar(
    top_subareas,
    x="ofertas",
    y="subarea",
    orientation="h",
    text="ofertas",
    title="Las 10 subáreas profesionales con más ofertas en Konzerta",
    labels={"ofertas": "Número de ofertas (anuncios)", "subarea": "Subárea o especialidad"},
    color="area",
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig_sub.update_layout(height=600, legend_title_text="Área Principal")
fig_sub.show()

### **Pregunta 2. Salarios según experiencia requerida y nivel laboral**

¿Cómo varían los salarios ofrecidos según la experiencia solicitada? Se seleccionan anuncios con salarios sin alertas y experiencia informada; se calculan medianas y promedios por rangos de meses. El nivel laboral se estudia como dimensión complementaria.

**Limitación:** en la última ejecución registrada había 414 ofertas con salario máximo sin alerta, pero solo **10** con salario y experiencia informada. La muestra combinada es muy pequeña y no permite generalizar una relación salarial al mercado laboral. La moneda y periodicidad de los importes no están confirmadas.

In [98]:
# ==========================================================
# SALARIOS Y EXPERIENCIA REQUERIDA
# ==========================================================

# 1. Seleccionar anuncios sin alertas salariales
salarios_validos = konzerta[
    (konzerta["estado_salario"] == "Registro sin alerta")
    & (konzerta["salario_max"].notna())
    & (konzerta["salario_max"] > 0)
].copy()

# 2. Seleccionar registros con experiencia válida
salarios_experiencia = salarios_validos[
    (salarios_validos["experiencia_meses"].notna())
    & (salarios_validos["experiencia_meses"] >= 0)
].copy()

# 3. Crear rangos de experiencia
salarios_experiencia["rango_experiencia"] = pd.cut(
    salarios_experiencia["experiencia_meses"],
    bins=[-0.01, 12, 36, 60, float("inf")],
    labels=[
        "0 a 12 meses",
        "13 a 36 meses",
        "37 a 60 meses",
        "Más de 60 meses"
    ]
)

# 4. Calcular estadísticas salariales por experiencia
resumen_salarios_experiencia = (
    salarios_experiencia
    .groupby(
        "rango_experiencia",
        observed=True
    )
    .agg(
        ofertas=("id", "size"),
        salario_max_promedio=("salario_max", "mean"),
        salario_max_mediana=("salario_max", "median"),
        salario_max_minimo=("salario_max", "min"),
        salario_max_maximo=("salario_max", "max")
    )
    .reset_index()
)

# 5. Mantener el análisis complementario por nivel laboral
resumen_salarios_nivel = (
    salarios_validos
    .groupby("nivel_laboral", dropna=False)
    .agg(
        ofertas=("id", "size"),
        salario_min_promedio=("salario_min", "mean"),
        salario_max_promedio=("salario_max", "mean"),
        salario_max_mediana=("salario_max", "median")
    )
    .reset_index()
    .sort_values(
        by="salario_max_mediana",
        ascending=False
    )
)

# 6. Mostrar resultados
print("PREGUNTA 2: SALARIOS SEGÚN EXPERIENCIA")
print("Ofertas con salario máximo:", len(salarios_validos))
print("Ofertas con salario y experiencia:", len(salarios_experiencia))

print("\nSALARIOS POR RANGO DE EXPERIENCIA")
display(resumen_salarios_experiencia.round(2))

print("\nSALARIOS POR NIVEL LABORAL")
display(resumen_salarios_nivel.round(2))

PREGUNTA 2: SALARIOS SEGÚN EXPERIENCIA
Ofertas con salario máximo: 414
Ofertas con salario y experiencia: 10

SALARIOS POR RANGO DE EXPERIENCIA


,rango_experiencia,ofertas,salario_max_promedio,salario_max_mediana,salario_max_minimo,salario_max_maximo
0,0 a 12 meses,2,825.00,825.0,750.0,900.0
1,13 a 36 meses,5,1138.00,900.0,640.0,2000.0
2,37 a 60 meses,3,1166.67,1200.0,850.0,1450.0



SALARIOS POR NIVEL LABORAL


,nivel_laboral,ofertas,salario_min_promedio,salario_max_promedio,salario_max_mediana
0,Jefe / Supervisor / Responsable,1,1300.00,1450.00,1450.0
5,Senior / Semi-Senior,10,NaN,1155.00,1050.0
1,Junior,15,888.89,1036.73,1000.0
3,Semi Sr,200,811.14,1051.12,989.0
2,Otro,4,675.00,912.50,900.0
4,Senior,2,787.50,900.00,900.0
7,Sin dato,181,794.43,1024.41,900.0
6,Sin Experiencia,1,550.00,750.00,750.0


### **Visualizaciones salariales: experiencia y nivel laboral**

El gráfico de barras compara la mediana del salario máximo entre rangos de experiencia; el diagrama de cajas permite examinar la dispersión salarial por nivel laboral. Los resultados son descriptivos y deben interpretarse con cautela debido a los pocos registros que informan simultáneamente salario y experiencia.

In [99]:
# ==========================================================
# VISUALIZACIONES SALARIALES
# ==========================================================

import plotly.express as px

# GRÁFICO 1: Salarios según experiencia requerida

if not resumen_salarios_experiencia.empty:

    fig_experiencia = px.bar(
        resumen_salarios_experiencia,
        x="rango_experiencia",
        y="salario_max_mediana",
        text="salario_max_mediana",
        color="rango_experiencia",
        title="Mediana salarial según experiencia requerida",
        labels={
            "rango_experiencia": "Experiencia requerida",
            "salario_max_mediana": "Mediana del salario máximo"
        },
        color_discrete_sequence=px.colors.qualitative.Safe
    )

    fig_experiencia.update_traces(
        texttemplate="%{text:,.2f}",
        textposition="outside"
    )

    fig_experiencia.update_layout(
        height=500,
        showlegend=False,
        xaxis_title="Rango de experiencia",
        yaxis_title="Mediana del salario máximo"
    )

    fig_experiencia.show()

else:
    print("No hay datos suficientes para graficar salarios por experiencia.")


# GRÁFICO 2: Distribución salarial según nivel laboral

if not salarios_validos.empty:

    fig_salarios = px.box(
        salarios_validos,
        x="nivel_laboral",
        y="salario_max",
        color="nivel_laboral",
        title="Distribución de salarios máximos por nivel laboral",
        labels={
            "nivel_laboral": "Nivel laboral",
            "salario_max": "Salario máximo ofrecido"
        },
        color_discrete_sequence=px.colors.qualitative.Safe
    )

    fig_salarios.update_layout(
        height=600,
        showlegend=False,
        xaxis_title="Nivel laboral",
        yaxis_title="Salario máximo ofrecido"
    )

    fig_salarios.show()

else:
    print("No hay datos salariales suficientes para el gráfico.")

### **Pregunta 3. Modalidades de trabajo por área profesional**

¿Qué modalidades de trabajo son más frecuentes y cómo cambian según el área?

Primero, analizaremos la distribución general de la variable `modalidad` para observar cuál es la opción predominante en el mercado laboral capturado por Konzerta.

In [100]:
# Distribución general de modalidades de trabajo
dist_modalidad = (
    konzerta["modalidad"].value_counts()
    .reset_index()
    .rename(columns={"count": "ofertas"})
)
dist_modalidad["porcentaje"] = (dist_modalidad["ofertas"] / len(konzerta) * 100).round(2)

display(dist_modalidad)

,modalidad,ofertas,porcentaje
0,Presencial,2942,97.71
1,Híbrido,59,1.96
2,Remoto,10,0.33


### **Relación entre la Modalidad y las 10 Áreas Principales**

Para comprender si la modalidad varía según la disciplina o tipo de profesión, cruzaremos estas variables enfocándonos en las 10 áreas con más anuncios.

In [101]:
# ==========================================================
# MODALIDADES DE TRABAJO POR ÁREA
# ==========================================================

# 1. Seleccionar las diez áreas con más ofertas
top_10_areas_nombres = (
    tabla_areas
    .sort_values("ofertas", ascending=False)
    .head(10)["area"]
    .tolist()
)

konzerta_top_areas = konzerta[
    konzerta["area"].isin(top_10_areas_nombres)
].copy()

# 2. Calcular la distribución porcentual por área
tabla_cruzada = (
    pd.crosstab(
        konzerta_top_areas["area"],
        konzerta_top_areas["modalidad"],
        normalize="index"
    ) * 100
).round(2).reset_index()

# 3. Convertir la tabla a formato largo
tabla_cruzada_larga = tabla_cruzada.melt(
    id_vars="area",
    var_name="modalidad",
    value_name="porcentaje"
)

# 4. Ordenar las áreas por número de ofertas
orden_areas = list(reversed(top_10_areas_nombres))

# 5. Crear el gráfico interactivo
fig_mod_area = px.bar(
    tabla_cruzada_larga,
    x="porcentaje",
    y="area",
    color="modalidad",
    orientation="h",
    category_orders={"area": orden_areas},
    title="Modalidades de trabajo en las 10 áreas con más ofertas",
    labels={
        "porcentaje": "Porcentaje de ofertas (%)",
        "area": "Área profesional",
        "modalidad": "Modalidad de trabajo"
    },
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig_mod_area.update_layout(
    height=650,
    barmode="stack",
    xaxis_range=[0, 100],
    legend_title_text="Modalidad"
)

fig_mod_area.update_traces(
    hovertemplate="%{y}: %{x:.2f}%<extra>%{fullData.name}</extra>"
)

fig_mod_area.show()

print("DISTRIBUCIÓN PORCENTUAL POR ÁREA")
display(tabla_cruzada)

DISTRIBUCIÓN PORCENTUAL POR ÁREA


modalidad,area,Híbrido,Presencial,Remoto
0,Abastecimiento y Logística,0.68,99.32,0.00
1,"Administración, Contabilidad y Finanzas",3.59,96.19,0.22
2,"Atención al Cliente, Call Center y Telemarketing",5.48,93.15,1.37
3,"Comercial, Ventas y Negocios",1.50,98.25,0.25
4,Gastronomía y Turismo,0.00,100.00,0.00
5,Ingeniería Civil y Construcción,0.00,100.00,0.00
6,Ingenierías,0.85,99.15,0.00
7,Oficios y Otros,0.00,100.00,0.00
8,Producción y Manufactura,0.00,100.00,0.00
9,"Tecnología, Sistemas y Telecomunicaciones",6.76,93.24,0.00


### **Pregunta 4. Distribución por provincia, área profesional y nivel laboral**

¿Cómo se distribuyen las oportunidades según ubicación, área y nivel laboral? Se calculan anuncios y porcentajes por provincia, cruces de provincia con área y con nivel laboral, y una tabla conjunta. Se comparan las cinco provincias con más anuncios; los resultados describen únicamente la muestra de Konzerta.

In [102]:
print(konzerta["provincia"].value_counts())

display(
    konzerta.loc[
        konzerta["provincia"].str.strip().str.lower() == "otra",
        ["titulo", "provincia", "localidad", "modalidad"]
    ].head(20)
)

provincia
Panamá            2358
Colón              148
Chiriquí           142
Panamá Oeste       121
Coclé               67
Veraguas            55
Herrera             47
Otra                42
Bocas del Toro      18
Los Santos          11
Darién               2
Name: count, dtype: Int64


,titulo,provincia,localidad,modalidad
58,Profesional de Seguridad e Higiene Industrial ...,Otra,Sin dato,Presencial
60,Profesional de Salud Ocupacional-Planta/Por ob...,Otra,Sin dato,Presencial
284,Vendedora de Boutique de Ballet,Otra,Sin dato,Presencial
288,Mercaderista,Otra,Sin dato,Presencial
307,Coordinador De Gestión De Calidad/ Regular,Otra,Sin dato,Presencial
501,Auxiliar De Procesos Biológicos/ Por Obra Dete...,Otra,Sin dato,Presencial
604,Secretaria Administrativa/Regular,Otra,Sin dato,Presencial
650,VENDEDOR DE PROYECTOS,Otra,Sin dato,Presencial
659,ASESOR DE RECURSOS HUMANOS Y DESARROLLO,Otra,Sin dato,Presencial
693,AYUDANTE GENERAL DE OPERADORES DE MAQUINAS DE ...,Otra,Sin dato,Presencial


In [103]:
# ==========================================================
# DISTRIBUCIÓN GEOGRÁFICA Y LABORAL
# ==========================================================

# 1. Distribución general por provincia
dist_provincia = (
    konzerta["provincia"]
    .value_counts()
    .reset_index()
    .rename(columns={"count": "ofertas"})
)

dist_provincia["porcentaje"] = (
    dist_provincia["ofertas"] / len(konzerta) * 100
).round(2)

print("OFERTAS POR PROVINCIA")
display(dist_provincia)


# 2. Distribución por provincia y área profesional
tabla_provincia_area = (
    konzerta
    .groupby(["provincia", "area"], dropna=False)
    .agg(ofertas=("id", "size"))
    .reset_index()
    .sort_values("ofertas", ascending=False)
)

print("\nOFERTAS POR PROVINCIA Y ÁREA")
display(tabla_provincia_area.head(20))


# 3. Distribución por provincia y nivel laboral
tabla_provincia_nivel = (
    konzerta
    .groupby(["provincia", "nivel_laboral"], dropna=False)
    .agg(ofertas=("id", "size"))
    .reset_index()
    .sort_values("ofertas", ascending=False)
)

print("\nOFERTAS POR PROVINCIA Y NIVEL LABORAL")
display(tabla_provincia_nivel.head(20))


# 4. Distribución conjunta
tabla_geografica_completa = (
    konzerta
    .groupby(
        ["provincia", "area", "nivel_laboral"],
        dropna=False
    )
    .agg(ofertas=("id", "size"))
    .reset_index()
    .sort_values("ofertas", ascending=False)
)

print("\nDISTRIBUCIÓN GEOGRÁFICA COMPLETA")
display(tabla_geografica_completa.head(20))

OFERTAS POR PROVINCIA


,provincia,ofertas,porcentaje
0,Panamá,2358,78.31
1,Colón,148,4.92
2,Chiriquí,142,4.72
3,Panamá Oeste,121,4.02
4,Coclé,67,2.23
5,Veraguas,55,1.83
6,Herrera,47,1.56
7,Otra,42,1.39
8,Bocas del Toro,18,0.6
9,Los Santos,11,0.37



OFERTAS POR PROVINCIA Y ÁREA


,provincia,area,ofertas
94,Panamá,"Comercial, Ventas y Negocios",555
91,Panamá,"Administración, Contabilidad y Finanzas",375
90,Panamá,Abastecimiento y Logística,217
107,Panamá,Oficios y Otros,174
114,Panamá,"Tecnología, Sistemas y Telecomunicaciones",136
93,Panamá,"Atención al Cliente, Call Center y Telemarketing",130
99,Panamá,Gastronomía y Turismo,122
108,Panamá,Producción y Manufactura,108
102,Panamá,Ingenierías,91
104,Panamá,Marketing y Publicidad,83



OFERTAS POR PROVINCIA Y NIVEL LABORAL


,provincia,nivel_laboral,ofertas
55,Panamá,Junior,771
57,Panamá,Semi Sr,645
58,Panamá,Senior,249
61,Panamá,Sin dato,205
59,Panamá,Senior / Semi-Senior,169
54,Panamá,Jefe / Supervisor / Responsable,137
56,Panamá,Otro,89
53,Panamá,Gerencia / Alta Gerencia / Dirección,71
27,Colón,Semi Sr,61
8,Chiriquí,Junior,53



DISTRIBUCIÓN GEOGRÁFICA COMPLETA


,provincia,area,nivel_laboral,ofertas
239,Panamá,"Comercial, Ventas y Negocios",Junior,201
241,Panamá,"Comercial, Ventas y Negocios",Semi Sr,138
217,Panamá,"Administración, Contabilidad y Finanzas",Junior,120
219,Panamá,"Administración, Contabilidad y Finanzas",Semi Sr,98
208,Panamá,Abastecimiento y Logística,Junior,81
326,Panamá,Oficios y Otros,Junior,65
210,Panamá,Abastecimiento y Logística,Semi Sr,61
242,Panamá,"Comercial, Ventas y Negocios",Senior,56
230,Panamá,"Atención al Cliente, Call Center y Telemarketing",Junior,55
328,Panamá,Oficios y Otros,Semi Sr,53


### **Visualización de la distribución por Provincia**

Creamos un gráfico interactivo para representar cómo se reparten las oportunidades de empleo en el territorio nacional.

In [104]:
fig_prov = px.bar(
    dist_provincia,
    x="porcentaje",
    y="provincia",
    orientation="h",
    text="porcentaje",
    title="Distribución porcentual de Ofertas de Empleo por Provincia en Panamá",
    labels={"porcentaje": "Porcentaje del total (%)", "provincia": "Provincia"},
    color="provincia",
    color_discrete_sequence=px.colors.qualitative.Prism
)

fig_prov.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_prov.update_layout(height=500, showlegend=False)
fig_prov.show()

In [105]:
# ==========================================================
# OFERTAS POR PROVINCIA Y ÁREA PROFESIONAL
# ==========================================================

# Seleccionar las cinco provincias con más anuncios
top_5_provincias = (
    dist_provincia
    .head(5)["provincia"]
    .tolist()
)

# Seleccionar las diez áreas profesionales con más anuncios
top_10_areas = (
    tabla_areas
    .sort_values("ofertas", ascending=False)
    .head(10)["area"]
    .tolist()
)

# Filtrar los registros
datos_geograficos = tabla_provincia_area[
    tabla_provincia_area["provincia"].isin(top_5_provincias)
    & tabla_provincia_area["area"].isin(top_10_areas)
].copy()

# Crear gráfico de barras agrupadas
fig_prov_area = px.bar(
    datos_geograficos,
    x="provincia",
    y="ofertas",
    color="area",
    barmode="group",
    title="Ofertas laborales por provincia y área profesional",
    labels={
        "provincia": "Provincia",
        "ofertas": "Cantidad de ofertas",
        "area": "Área profesional"
    },
    category_orders={
        "provincia": top_5_provincias
    },
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig_prov_area.update_layout(
    height=650,
    xaxis_title="Provincia",
    yaxis_title="Número de ofertas",
    legend_title_text="Área profesional"
)

fig_prov_area.show()

In [106]:
# ==========================================================
# NIVEL LABORAL POR PROVINCIA
# ==========================================================

# Seleccionar las cinco provincias con más ofertas
top_5_provincias = (
    dist_provincia
    .head(5)["provincia"]
    .tolist()
)

# Filtrar las ofertas de esas provincias
datos_nivel_provincia = tabla_provincia_nivel[
    tabla_provincia_nivel["provincia"].isin(top_5_provincias)
].copy()

# Calcular el total de ofertas por provincia
datos_nivel_provincia["total_provincia"] = (
    datos_nivel_provincia
    .groupby("provincia")["ofertas"]
    .transform("sum")
)

# Calcular la participación de cada nivel laboral
datos_nivel_provincia["porcentaje"] = (
    datos_nivel_provincia["ofertas"]
    / datos_nivel_provincia["total_provincia"] * 100
)

# Crear gráfico de barras apiladas
fig_nivel_provincia = px.bar(
    datos_nivel_provincia,
    x="porcentaje",
    y="provincia",
    color="nivel_laboral",
    orientation="h",
    title="Distribución de niveles laborales por provincia",
    labels={
        "porcentaje": "Porcentaje de ofertas (%)",
        "provincia": "Provincia",
        "nivel_laboral": "Nivel laboral"
    },
    category_orders={
        "provincia": list(reversed(top_5_provincias))
    },
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig_nivel_provincia.update_layout(
    height=600,
    barmode="stack",
    xaxis_range=[0, 100],
    legend_title_text="Nivel laboral"
)

fig_nivel_provincia.update_traces(
    hovertemplate="%{y}: %{x:.2f}%<extra>%{fullData.name}</extra>"
)

fig_nivel_provincia.show()

### **Pregunta 5. Evolución semanal de las ofertas (julio–octubre de 2026)**

Las ofertas se filtran por su fecha de publicación entre el 1 de julio y el 31 de octubre de 2026 y se agrupan en semanas que empiezan el lunes. Se conservan semanas sin registros dentro del intervalo observado. **La extracción del archivo se realizó el 16 de octubre de 2026**: no corresponde interpretar las fechas posteriores a esa extracción como semanas sin demanda; el gráfico se limita a la ventana realmente observada.

In [107]:
# ==========================================================
# EVOLUCIÓN SEMANAL OBSERVADA
# ==========================================================
fechas_publicacion = pd.to_datetime(konzerta["fecha_publicacion"], errors="coerce")
fecha_inicio = pd.Timestamp("2026-07-01")
fecha_fin_teorica = pd.Timestamp("2026-10-31")
# Fecha de obtención documentada en la portada del proyecto
fecha_extraccion = pd.Timestamp("2026-10-16")
fecha_fin_observada = min(fecha_fin_teorica, fecha_extraccion)
ofertas_periodo = konzerta.loc[
    fechas_publicacion.between(fecha_inicio, fecha_fin_observada + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1))
].copy()
ofertas_periodo["fecha_publicacion"] = fechas_publicacion.loc[ofertas_periodo.index]
ofertas_periodo["semana_inicio"] = (
    ofertas_periodo["fecha_publicacion"].dt.to_period("W-SUN").dt.start_time
)
semanas_observadas = pd.date_range(
    start=fecha_inicio.to_period("W-SUN").start_time,
    end=fecha_fin_observada.to_period("W-SUN").start_time,
    freq="W-MON"
)
tendencia_semanal = (
    ofertas_periodo.groupby("semana_inicio").size()
    .reindex(semanas_observadas, fill_value=0)
    .rename_axis("semana_inicio").reset_index(name="ofertas")
)
tendencia_semanal["semana_parcial"] = (
    (tendencia_semanal["semana_inicio"] < fecha_inicio)
    | (tendencia_semanal["semana_inicio"] + pd.Timedelta(days=6) > fecha_fin_observada)
)
print("Ofertas en el período observado:", len(ofertas_periodo))
print("Último día de observación:", fecha_fin_observada.date())
print("Semanas (incluye semanas parciales):", len(tendencia_semanal))
display(tendencia_semanal)

Ofertas en el período observado: 3011
Último día de observación: 2026-10-16
Semanas (incluye semanas parciales): 16


,semana_inicio,ofertas,semana_parcial
0,2026-06-29,0,True
1,2026-07-06,2,False
2,2026-07-13,40,False
3,2026-07-20,32,False
4,2026-07-27,30,False
5,2026-08-03,35,False
6,2026-08-10,275,False
7,2026-08-17,296,False
8,2026-08-24,281,False
9,2026-08-31,344,False


### **Visualización de la tendencia semanal**

El gráfico presenta el volumen semanal observado desde julio hasta la fecha de extracción, con marcadores para las semanas. La primera y última semana pueden estar incompletas, por lo que sus cantidades no son directamente comparables con semanas completas.

In [108]:
# ==========================================================
# GRÁFICO DE EVOLUCIÓN SEMANAL
# ==========================================================
fig_linea = px.line(
    tendencia_semanal,
    x="semana_inicio",
    y="ofertas",
    markers=True,
    title="Evolución semanal de ofertas de empleo (julio–octubre de 2026)",
    labels={"semana_inicio": "Semana de publicación", "ofertas": "Número de ofertas"}
)
fig_linea.update_traces(
    line=dict(width=3), marker=dict(size=8),
    hovertemplate="Semana del %{x|%d/%m/%Y}<br>Ofertas: %{y}<extra></extra>"
)
fig_linea.update_layout(height=500, hovermode="x unified")
fig_linea.show()
print("Nota: la primera y última semana pueden estar incompletas.")

Nota: la primera y última semana pueden estar incompletas.


## **Capa oro: tablas agregadas para el dashboard**

Se construyen tablas agregadas a partir de la capa plata. El dashboard de Streamlit **solo lee archivos de la capa oro**. Las cifras representan anuncios del archivo, no todo el mercado laboral. Las agregaciones de salarios incluyen exclusivamente registros sin alertas y las de experiencia se interpretan con precaución.

In [109]:
# ==========================================================
# GENERACIÓN Y EXPORTACIÓN DE TABLAS AGREGADAS
# ==========================================================
ORO.mkdir(parents=True, exist_ok=True)

oro_areas = (
    konzerta.groupby(["area", "subarea"], dropna=False)
    .agg(ofertas=("id", "size"), vacantes=("vacantes", "sum"))
    .reset_index()
    .sort_values("ofertas", ascending=False)
)
oro_modalidades = (
    konzerta.groupby(["area", "modalidad"], dropna=False)
    .agg(ofertas=("id", "size"))
    .reset_index()
)
oro_geografia = (
    konzerta.groupby(["provincia", "area", "nivel_laboral"], dropna=False)
    .agg(ofertas=("id", "size"), vacantes=("vacantes", "sum"))
    .reset_index()
)
oro_salarios_experiencia = resumen_salarios_experiencia.copy()
oro_salarios_nivel = resumen_salarios_nivel.copy()
oro_tendencia_semanal = tendencia_semanal.copy()

tablas_oro = {
    "oro_areas_subareas.csv": oro_areas,
    "oro_modalidades_area.csv": oro_modalidades,
    "oro_geografia.csv": oro_geografia,
    "oro_salarios_experiencia.csv": oro_salarios_experiencia,
    "oro_salarios_nivel.csv": oro_salarios_nivel,
    "oro_tendencia_semanal.csv": oro_tendencia_semanal,
}
for nombre, tabla in tablas_oro.items():
    tabla.to_csv(ORO / nombre, index=False, encoding="utf-8-sig")
    print(f"{nombre}: {len(tabla)} filas")
print("Capa oro generada:", ORO)

oro_areas_subareas.csv: 174 filas
oro_modalidades_area.csv: 40 filas
oro_geografia.csv: 452 filas
oro_salarios_experiencia.csv: 3 filas
oro_salarios_nivel.csv: 8 filas
oro_tendencia_semanal.csv: 16 filas
Capa oro generada: /content/drive/MyDrive/Proyecto_Konzerta/datos/oro


## **Validación de la capa oro**

Se comprueba que las tablas agregadas reproduzcan el total de anuncios cuando corresponda, que existan los archivos de salida y que la información salarial sea presentada como una muestra restringida.

In [110]:
# ==========================================================
# VALIDACIÓN DE TOTALES Y ARCHIVOS
# ==========================================================
total_ofertas = len(konzerta)
assert int(oro_areas["ofertas"].sum()) == total_ofertas
assert int(oro_modalidades["ofertas"].sum()) == total_ofertas
assert int(oro_geografia["ofertas"].sum()) == total_ofertas
assert all((ORO / nombre).is_file() for nombre in tablas_oro)
print("Validación correcta: las agregaciones principales coinciden con", total_ofertas, "anuncios.")
print("Salarios con experiencia:", len(salarios_experiencia), "ofertas (interpretar con cautela).")

Validación correcta: las agregaciones principales coinciden con 3011 anuncios.
Salarios con experiencia: 10 ofertas (interpretar con cautela).
